# 04. CatBoost ranking

Обучение на снимках сентября–декабря 2000 и оценка на январском validation. Retrieval, признаки и evaluator используются из общего пакета. Основная модель сравнивается с retrieval-baselines и альтернативными наборами признаков.

## 1. Данные и общие функции

fit_retrieval обучает модели только на переданной истории. Функции подготовки снимков и рекомендаций используются совместно с финальным тестом и приложением.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from movielens_recsys.data import load_movielens

if (Path.cwd() / "../movielens-reccomender/configs/experiment.json").exists():
    ROOT = (Path.cwd() / "../movielens-reccomender").resolve()
elif (Path.cwd() / "configs/experiment.json").exists():
    ROOT = Path.cwd()
else:
    ROOT = (Path.cwd() / "ml-1m/movielens-reccomender").resolve()

config = json.loads((ROOT / "configs/experiment.json").read_text(encoding="utf-8"))
ratings, movies, users = load_movielens(ROOT / config["data_dir"])
SEED = config["seed"]
retrieval_comparison = pd.read_csv(ROOT / "reports/retrieval/validation.csv")
display(retrieval_comparison.query("k == 250"))

from movielens_recsys.retrieval import (
    fit_retrieval,
    recommend_als,
    recommend_item_knn,
    recommend_popular,
)
from movielens_recsys.ranking import build_snapshot, split_snapshot


,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
18,MostPopular,250,0.036071,0.438946,0.808300,0.202682,0.304513,250.0,506
19,ALS,250,0.035565,0.513445,0.853755,0.228784,0.505710,250.0,506
20,ItemKNN,250,0.038245,0.460495,0.814229,0.218320,0.610114,250.0,506
21,Union_RRF+Popular,250,0.038427,0.498089,0.843874,0.226536,0.403208,250.0,506
22,Union_ALS_first+Popular,250,0.038427,0.498089,0.843874,0.229521,0.403208,250.0,506
23,Union_ALS150_KNN100+Popular,250,0.038877,0.519402,0.849802,0.233918,0.455682,250.0,506


## 2. Временные окна

Сентябрь-декабрь - train, январь - validation. История всегда строго до начала месяца, метки - внутри целевого окна. Февраль не участвует в обучении и выборе признаков.

In [2]:
snapshot_plan = pd.DataFrame([
    {"role": "train", "start": "2000-09-01", "end": "2000-10-01"},
    {"role": "train", "start": "2000-10-01", "end": "2000-11-01"},
    {"role": "train", "start": "2000-11-01", "end": "2000-12-01"},
    {"role": "train", "start": "2000-12-01", "end": "2001-01-01"},
    {"role": "validation", "start": "2001-01-01", "end": "2001-02-01"},
])
snapshot_plan["start"] = pd.to_datetime(snapshot_plan["start"], utc=True)
snapshot_plan["end"] = pd.to_datetime(snapshot_plan["end"], utc=True)
display(snapshot_plan)

,role,start,end
0,train,2000-09-01 00:00:00+00:00,2000-10-01 00:00:00+00:00
1,train,2000-10-01 00:00:00+00:00,2000-11-01 00:00:00+00:00
2,train,2000-11-01 00:00:00+00:00,2000-12-01 00:00:00+00:00
3,train,2000-12-01 00:00:00+00:00,2001-01-01 00:00:00+00:00
4,validation,2001-01-01 00:00:00+00:00,2001-02-01 00:00:00+00:00


### 2.1. Сентябрьский снимок

История и будущие ответы разделены по границе 1 сентября; та же схема повторяется для остальных месяцев.

In [3]:
# История и ответы сентябрьского снимка.

start = snapshot_plan.iloc[0]["start"]
end = snapshot_plan.iloc[0]["end"]
history = ratings[ratings["datetime"] < start].copy()
target_window = ratings[
    (ratings["datetime"] >= start) & (ratings["datetime"] < end)
].copy()
positive_history = history[history["rating"] >= config["positive_threshold"]]
seen_movies = history.groupby("user_id")["movie_id"].apply(set).to_dict()
ground_truth = (
    target_window[target_window["rating"] >= config["positive_threshold"]]
    .groupby("user_id")["movie_id"].apply(set).to_dict()
)
eligible_users = sorted(set(ground_truth) & set(history["user_id"]))
catalog = set(history["movie_id"])
assert history["datetime"].max() < start
print("Сентябрь: history", len(history), "eligible users", len(eligible_users))

Сентябрь: history 405762 eligible users 242


## 3. Подготовка снимка

Общие функции выполняют разделение времени, подбор кандидатов и вычисление признаков. Для каждого месяца используются его собственные retrieval-модели.

### 3.1. Отделяем историю от ответов
`split_snapshot` импортирована из `src`. Она возвращает прошлое для моделей и следующий месяц для target.

### 3.2. Получаем кандидатов и retrieval-признаки
`build_snapshot_candidates` импортирована из `src`. Она обучает retrieval на прошлом месяца, берёт ALS150 + KNN100 и дополняет список Popular до 250.

## 4. Target и признаки

Число и средняя оценка пользователя/фильма вычисляются из history. Target берётся из целевого месяца. Group_id объединяет дату и user_id. Одна строка — один кандидат; пропущенные retrieval положительные фильмы в кандидаты не добавляются.

### Профиль пользователя

Пол, возрастная группа и профессия добавляются только в эксперимент с 17 признаками. age=25 — код группы 25–34, а не точный возраст. Категории передаются через cat_features. Профиль считается статичным и доступным на cutoff; истории его изменений в MovieLens нет. ZIP code не используется.

### 3.4. Собираем один снимок целиком
`build_snapshot` импортирована из `src` и последовательно вызывает три шага выше. В ноутбуке остаётся только вызов этой функции.

### 4.2. Train за четыре месяца

Цикл вызывает build_snapshot для сентября, октября, ноября и декабря, затем объединяет таблицы. Retrieval каждого снимка обучен на его собственном прошлом.

In [4]:
train_periods = [
    ("2000-09-01", "2000-10-01"),
    ("2000-10-01", "2000-11-01"),
    ("2000-11-01", "2000-12-01"),
    ("2000-12-01", "2001-01-01"),
]

train_snapshots = []

for start_string, end_string in train_periods:
    start = pd.Timestamp(start_string, tz="UTC")
    end = pd.Timestamp(end_string, tz="UTC")

    snapshot_frame = build_snapshot(
        ratings=ratings,
        users=users,
        start=start,
        end=end,
        config=config,
        seed=SEED,
    )

    train_snapshots.append(snapshot_frame)

train_frame = pd.concat(
    train_snapshots,
    ignore_index=True,
)

print("Train:", train_frame.shape)
display(train_frame.head())


Train: (311750, 22)


,snapshot_time,user_id,movie_id,als_score,knn_score,als_rank,knn_rank,popular_rank,from_als,from_knn,...,knn_score_missing,user_rating_count,user_mean_rating,item_rating_count,item_mean_rating,gender,age,occupation,target,group_id
0,2000-09-01 00:00:00+00:00,3353,2762,0.420438,6.757295,1,7,3,1,1,...,0,32,4.375,1119,4.403038,M,35,17,0,2000-09-01_user_3353
1,2000-09-01 00:00:00+00:00,3353,110,0.417393,7.534476,2,2,13,1,1,...,0,32,4.375,931,4.286788,M,35,17,1,2000-09-01_user_3353
2,2000-09-01 00:00:00+00:00,3353,50,0.411079,7.197207,3,3,24,1,1,...,0,32,4.375,733,4.515689,M,35,17,0,2000-09-01_user_3353
3,2000-09-01 00:00:00+00:00,3353,1617,0.335411,7.872994,4,1,12,1,1,...,0,32,4.375,951,4.232387,M,35,17,0,2000-09-01_user_3353
4,2000-09-01 00:00:00+00:00,3353,36,0.292194,1.563801,5,62,222,1,1,...,0,32,4.375,342,3.944444,M,35,17,0,2000-09-01_user_3353


### 4.3. Создаём январский validation
Те же признаки и та же политика кандидатов. Отличается дата: история до января, target за январь. Train и validation не смешиваем.

In [5]:
validation_frame = build_snapshot(
    ratings=ratings,
    users=users,
    start=pd.Timestamp("2001-01-01", tz="UTC"),
    end=pd.Timestamp("2001-02-01", tz="UTC"),
    config=config,
    seed=SEED,
)
print("Validation:", validation_frame.shape)
display(validation_frame.head())

Validation: (126500, 22)


,snapshot_time,user_id,movie_id,als_score,knn_score,als_rank,knn_rank,popular_rank,from_als,from_knn,...,knn_score_missing,user_rating_count,user_mean_rating,item_rating_count,item_mean_rating,gender,age,occupation,target,group_id
0,2001-01-01 00:00:00+00:00,1,318,0.398648,8.649085,1,3,9,1,1,...,0,40,4.175,2088,4.558908,F,1,10,0,2001-01-01_user_1
1,2001-01-01 00:00:00+00:00,1,593,0.367448,7.442471,2,6,4,1,1,...,0,40,4.175,2432,4.356497,F,1,10,0,2001-01-01_user_1
2,2001-01-01 00:00:00+00:00,1,1259,0.366303,6.267778,3,8,27,1,1,...,0,40,4.175,1699,4.107122,F,1,10,0,2001-01-01_user_1
3,2001-01-01 00:00:00+00:00,1,1,0.322076,8.171663,4,4,18,1,1,...,0,40,4.175,1967,4.146924,F,1,10,1,2001-01-01_user_1
4,2001-01-01 00:00:00+00:00,1,3471,0.315892,2.751456,5,54,58,1,1,...,0,40,4.175,1330,4.005263,F,1,10,0,2001-01-01_user_1


## 5. Группы и Pool

Строки одной группы идут подряд. Группа — пользователь в конкретном месяце, внутри которой CatBoost сравнивает кандидатов. Группы с одинаковыми метками не дают полезного сигнала для сравнения и исключаются только из train fitting. В validation остаются все группы. Имена и порядок признаков одинаковые в train и validation.

- min=0, max=1: в группе есть положительные и неположительные кандидаты.
- min=max=0: все метки нулевые.
- min=max=1: все метки положительные.

### 5.1. Состав групп

positive_share — доля единиц среди строк; groups_without_positive — группы со всеми нулевыми метками. Таблица показывает количество групп и строк, исключённых только из обучения.

In [7]:
statistics = []

for name, frame in [("train", train_frame), ("validation", validation_frame)]:
    stats = frame.groupby("group_id")["target"].agg(["min", "max"])
    useful = stats[stats["min"] < stats["max"]].index
    useful_rows = frame["group_id"].isin(useful)

    statistics.append({
        "dataset": name,
        "rows": len(frame),
        "groups": len(stats),
        "positive_share": frame["target"].mean(),
        "groups_without_positive": int((stats["max"] == 0).sum()),
        "groups_all_positive": int((stats["min"] == 1).sum()),
        "useful_groups": len(useful),
        "excluded_from_fitting_groups": len(stats) - len(useful) if name == "train" else 0,
        "excluded_from_fitting_rows": int((~useful_rows).sum()) if name == "train" else 0,
    })

group_summary = pd.DataFrame(statistics)
display(group_summary)

group_stats = train_frame.groupby("group_id")["target"].agg(["min", "max"])
display(group_stats.head(10))

,dataset,rows,groups,positive_share,groups_without_positive,groups_all_positive,useful_groups,excluded_from_fitting_groups,excluded_from_fitting_rows
0,train,311750,1247,0.043176,219,0,1028,219,54750
1,validation,126500,506,0.038877,76,0,430,0,0


,min,max
group_id,,
2000-09-01_user_3353,0,1
2000-09-01_user_3355,0,1
2000-09-01_user_3359,0,1
2000-09-01_user_3363,0,1
2000-09-01_user_3364,0,1
2000-09-01_user_3366,0,1
2000-09-01_user_3367,0,1
2000-09-01_user_3368,0,1
2000-09-01_user_3370,0,1


### 5.2. Основной набор признаков

FEATURES содержит 11 числовых признаков из конфига. Дополнительные rank/missing и демография остаются в таблицах для сравнения в разделе 7.5; основной Pool их не использует.

In [ ]:
BASE_FEATURES = list(config["ranking"]["features"])

# Дополнительные признаки для отдельного эксперимента.
EXTRA_FEATURES = ["popular_rank", "als_score_missing", "knn_score_missing"]

# gender, age и occupation - категории, а не числовые шкалы.
DEMOGRAPHIC_FEATURES = ["gender", "age", "occupation"]
# Основной вариант: только 11 выбранных признаков из конфига.
FEATURES = list(BASE_FEATURES)
CAT_FEATURES = list(config["ranking"]["cat_features"])
assert len(FEATURES) == 11
assert CAT_FEATURES == []
assert len(FEATURES) == len(set(FEATURES)), "В FEATURES есть повторяющиеся имена"

# Проверка наличия выбранных признаков в train и validation.
for name, frame in [("train", train_frame), ("validation", validation_frame)]:
    missing_features = [feature for feature in FEATURES if feature not in frame.columns]
    assert not missing_features, (name, missing_features)

# Для ранжирования нужны группы с разными метками.
group_stats = train_frame.groupby("group_id")["target"].agg(["min", "max"])
useful_groups = group_stats[group_stats["min"] < group_stats["max"]].index
train_fit = train_frame[train_frame["group_id"].isin(useful_groups)].copy()
train_fit = train_fit.sort_values(["group_id", "movie_id"])
validation_frame = validation_frame.sort_values(["group_id", "movie_id"])
from catboost import Pool
train_pool = Pool(train_fit[FEATURES], label=train_fit["target"], group_id=train_fit["group_id"], cat_features=CAT_FEATURES)
validation_pool = Pool(validation_frame[FEATURES], label=validation_frame["target"], group_id=validation_frame["group_id"], cat_features=CAT_FEATURES)

## 6. CatBoostRanker

Основной вариант: YetiRank, depth=6, learning_rate=0.05, 66 деревьев, seed=42. Число деревьев ранее выбрано early stopping на январе. Встроенная NDCG отслеживает качество внутри candidate Pool; для полного end-to-end сравнения используется внешний evaluator. Отдельный эксперимент подбора параметров представлен в ноутбуке 06.

In [9]:
from catboost import CatBoostRanker

# Все выбранные параметры находятся в конфиге.
ranker = CatBoostRanker(
    **config["ranking"]["parameters"],
    random_seed=SEED,
    verbose=50,
)
# Выбор числа деревьев уже сделан по validation. Основной fit его не меняет.
ranker.fit(train_pool)
assert ranker.tree_count_ == config["ranking"]["parameters"]["iterations"]


0:	total: 334ms	remaining: 21.7s


50:	total: 7.99s	remaining: 2.35s


65:	total: 10.9s	remaining: 0us


## 7. Сравнение моделей

Списки кандидатов сортируются по CatBoost score DESC, movie_id ASC и оцениваются против полного январского ground truth. Сравниваются MostPopular, ALS, ItemKNN, исходный Union и CatBoost. Recall полного пула не меняется от ранжирования; NDCG@10 отражает порядок.

### 7.1. Предсказываем score и строим списки
 Копия validation не меняет признаки исходной таблицы. Получаем score каждого кандидата и сортируем фильмы отдельно для каждого пользователя. Сохраняем все 250, чтобы проверить потолок retrieval.

In [10]:

prediction_frame = validation_frame.copy()
prediction_frame["catboost_score"] = ranker.predict(prediction_frame[FEATURES])
prediction_frame = prediction_frame.sort_values(
    ["user_id", "catboost_score", "movie_id"],
    ascending=[True, False, True],
)

ranked_recommendations = (
    prediction_frame.groupby("user_id")["movie_id"].apply(list).to_dict()
)

display(prediction_frame[["user_id", "movie_id", "catboost_score", "target"]].head(10))

,user_id,movie_id,catboost_score,target
0,1,318,1.694535,0
1,1,593,1.648493,0
3,1,1,1.595330,1
8,1,1196,1.572584,0
15,1,1198,1.540324,0
9,1,527,1.537131,1
2,1,1259,1.411528,0
16,1,2716,1.198503,0
83,1,296,1.140325,0
36,1,2858,1.117564,0


### 7.2. Январский ground truth

Отдельные validation-переменные относятся к январю; демонстрационный снимок в разделе 2 — к сентябрю. Ground truth сохраняет все положительные январские фильмы, включая пропущенные retrieval.

In [11]:
validation_start = pd.Timestamp(config["split"]["validation_start"])
validation_end = pd.Timestamp(config["split"]["test_start"])
validation_history, validation_target = split_snapshot(
    ratings, validation_start, validation_end,
)
validation_ground_truth = (
    validation_target[validation_target["rating"] >= config["positive_threshold"]]
    .groupby("user_id")["movie_id"].apply(set).to_dict()
)
validation_users = sorted(
    set(validation_ground_truth) & set(validation_history["user_id"])
)
validation_catalog = set(validation_history["movie_id"])
validation_seen = validation_history.groupby("user_id")["movie_id"].apply(set).to_dict()

assert set(ranked_recommendations) == set(validation_users)
for user_id in validation_users:
    recs = ranked_recommendations[user_id]
    assert len(recs) == len(set(recs))
    assert set(recs).isdisjoint(validation_seen[user_id])
    assert set(recs).issubset(validation_catalog)

### 7.3. С чем сравниваем CatBoost
Получим отдельные январские MostPopular, ALS и KNN. Для честного сравнения с исходным порядком восстановим тот же union ALS150 + KNN100 + Popular-добивку.
Это ещё один fit retrieval на январской истории для контроля, а не обучение CatBoost на январских метках. Параметры и seed такие же, как при создании validation_frame.

In [12]:
validation_retrieval = fit_retrieval(
    validation_history,
    positive_threshold=config["positive_threshold"],
    seed=SEED,
    n_neighbors=config["retrieval"]["item_knn_neighbors"],
    als_parameters=config["retrieval"]["als_parameters"],
)
popular_recommendations = {}
als_recommendations = {}
knn_recommendations = {}
original_union_recommendations = {}

for user_id in validation_users:
    popular = recommend_popular(user_id, config["retrieval"]["max_candidates"], validation_retrieval)
    als = recommend_als(user_id, config["retrieval"]["max_candidates"], validation_retrieval)
    knn = recommend_item_knn(user_id, config["retrieval"]["max_candidates"], validation_retrieval)
    popular_recommendations[user_id] = popular
    als_recommendations[user_id] = als
    knn_recommendations[user_id] = knn

    candidates = []
    for movie_id in als[:config["retrieval"]["als_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in knn[:config["retrieval"]["item_knn_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in popular:
        if len(candidates) >= config["retrieval"]["max_candidates"]:
            break
        if movie_id not in candidates:
            candidates.append(movie_id)
    original_union_recommendations[user_id] = candidates
    assert set(candidates) == set(ranked_recommendations[user_id])

### 7.4. Один evaluator для всех моделей
Формулы из 03 вынесены в `src/movielens_recsys/evaluation/metrics.py`, чтобы не копировать их сюда. NDCG использует полный ground truth. Сначала сравниваем все модели при одном K, затем при следующем.
Recall@250 исходного union и CatBoost должен совпасть: ranker меняет порядок, а не состав кандидатов.

In [13]:
from movielens_recsys.evaluation import evaluate_recommendations

models = {
    "MostPopular": popular_recommendations,
    "ALS": als_recommendations,
    "ItemKNN": knn_recommendations,
    "Union_ALS150_KNN100+Popular": original_union_recommendations,
    "CatBoost": ranked_recommendations,
}

results = []
for k in [5, 10, 20, 250]:
    for model_name, recommendations in models.items():
        _, result = evaluate_recommendations(
            recommendations, validation_ground_truth, validation_users,
            validation_catalog, model_name=model_name, k=k,
        )
        results.append(result)

ranking_comparison = pd.DataFrame(results)
display(ranking_comparison)

# Проверка неизменности Recall полного пула.
recall_at_250 = ranking_comparison[ranking_comparison["k"] == 250].set_index("model")["recall"]
assert np.isclose(recall_at_250["CatBoost"], recall_at_250["Union_ALS150_KNN100+Popular"])

# Oracle NDCG@10: лучший порядок, возможный при текущих кандидатах.
import math
oracle_scores = []
for user_id in validation_users:
    relevant = validation_ground_truth[user_id]
    available_hits = len(set(ranked_recommendations[user_id]) & relevant)
    best_dcg = sum(1 / math.log2(rank + 1) for rank in range(1, min(available_hits, 10) + 1))
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), 10) + 1))
    oracle_scores.append(best_dcg / idcg)
print("Oracle NDCG@10:", np.mean(oracle_scores))

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
0,MostPopular,5,0.099605,0.027657,0.264822,0.110436,0.023382,5.0,506
1,ALS,5,0.104743,0.040495,0.322134,0.115278,0.105492,5.0,506
2,ItemKNN,5,0.120949,0.039562,0.322134,0.132914,0.038608,5.0,506
3,Union_ALS150_KNN100+Popular,5,0.104743,0.040495,0.322134,0.115278,0.105492,5.0,506
4,CatBoost,5,0.121739,0.039396,0.324111,0.136046,0.036705,5.0,506
5,MostPopular,10,0.091304,0.049683,0.359684,0.108800,0.036705,10.0,506
6,ALS,10,0.092688,0.075491,0.446640,0.116128,0.146003,10.0,506
7,ItemKNN,10,0.107708,0.061298,0.387352,0.128906,0.060087,10.0,506
8,Union_ALS150_KNN100+Popular,10,0.092688,0.075491,0.446640,0.116128,0.146003,10.0,506
9,CatBoost,10,0.106126,0.064781,0.413043,0.129987,0.060903,10.0,506


Oracle NDCG@10: 0.7039966221164272


### Сохранение модели

Обученный CatBoost сохраняется для финального теста и Streamlit. Метрики остаются в таблице сравнения выше.

In [14]:
# Создаём папку для сохранённой модели
model_dir = ROOT / "artifacts" / "ranking"
model_dir.mkdir(parents=True, exist_ok=True)

# Сохраняем обученный CatBoost
model_path = model_dir / "catboost_base.cbm"
ranker.save_model(model_path)

print("Модель сохранена:", model_path.name)

Модель сохранена: catboost_base.cbm


### 7.5. Отдельные эксперименты с дополнительными признаками
Основная модель с 11 признаками уже обучена в разделе 6 и здесь используется повторно. Варианты 14 и 17 обучаются отдельно с прежним лимитом 300 деревьев и early stopping. Они не заменяют ranker, FEATURES или конфиг основного решения.
Пул кандидатов и 506 пользователей для всех вариантов одинаковые. Сравниваем внешние NDCG/Recall/Precision@10. Число деревьев основной модели уже выбрано прежним экспериментом, поэтому 66 фиксировано, а не оптимизируется повторно. Февраль не используется.

In [15]:
# Одинаковые даты, кандидаты, гиперпараметры и evaluator.
# Меняется только набор признаков.
feature_sets = {
    "CatBoost_base": BASE_FEATURES,
    "CatBoost_rank_missing": BASE_FEATURES + EXTRA_FEATURES,
    "CatBoost_rank_missing_demography": BASE_FEATURES + EXTRA_FEATURES + DEMOGRAPHIC_FEATURES,
}
ablation_results = []

for experiment_name, columns in feature_sets.items():
    categorical_columns = [column for column in DEMOGRAPHIC_FEATURES if column in columns]

    if experiment_name == config["ranking"]["selected_model"]:
        # Эта модель уже обучена в разделе 6; не обучаем её второй раз.
        experiment_model = ranker
    else:
        experiment_train_pool = Pool(
            train_fit[columns], label=train_fit["target"],
            group_id=train_fit["group_id"], cat_features=categorical_columns,
        )
        experiment_validation_pool = Pool(
            validation_frame[columns], label=validation_frame["target"],
            group_id=validation_frame["group_id"], cat_features=categorical_columns,
        )
        experiment_parameters = dict(config["ranking"]["parameters"])
        experiment_parameters["iterations"] = config["ranking"]["ablation"]["max_iterations"]
        experiment_model = CatBoostRanker(
            **experiment_parameters,
            random_seed=SEED, verbose=False,
        )
        experiment_model.fit(
            experiment_train_pool, eval_set=experiment_validation_pool,
            early_stopping_rounds=config["ranking"]["ablation"]["early_stopping_rounds"],
        )

    scored = validation_frame[["user_id", "movie_id"]].copy()
    # Предсказанные оценки CatBoost для кандидатов в validation.
    scored["score"] = experiment_model.predict(validation_frame[columns])
    scored = scored.sort_values(
        ["user_id", "score", "movie_id"], ascending=[True, False, True],
    )
    recommendations = scored.groupby("user_id")["movie_id"].apply(list).to_dict()
    for user_id in validation_users:
        assert set(recommendations[user_id]) == set(original_union_recommendations[user_id])

    _, result = evaluate_recommendations(
        recommendations, validation_ground_truth, validation_users,
        validation_catalog, model_name=experiment_name, k=10,
    )
    result["feature_count"] = len(columns)
    # Для основного fit без eval_set get_best_iteration() вернул бы -1.
    # Записываем исходную выбранную validation-итерацию отдельно от числа деревьев.
    if experiment_name == config["ranking"]["selected_model"]:
        result["best_iteration"] = config["ranking"]["selection"]["selected_best_iteration_zero_based"]
    else:
        result["best_iteration"] = experiment_model.get_best_iteration()
    result["tree_count"] = experiment_model.tree_count_
    ablation_results.append(result)

feature_ablation = pd.DataFrame(ablation_results)
display(feature_ablation)

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users,feature_count,best_iteration,tree_count
0,CatBoost_base,10,0.106126,0.064781,0.413043,0.129987,0.060903,10.0,506,11,65,66
1,CatBoost_rank_missing,10,0.105731,0.062175,0.401186,0.126563,0.054921,10.0,506,14,21,22
2,CatBoost_rank_missing_demography,10,0.106719,0.063765,0.411067,0.129402,0.060903,10.0,506,17,118,119


### Выбор модели

NDCG@10: 0.129987 у 11 базовых признаков, 0.126563 у 14 признаков, 0.129402 у 17. Дополнительные признаки не улучшили основную метрику; исходная модель с 11 признаками сохранена основной.

CatBoost улучшил NDCG относительно исходного порядка ALS-first, но пока не показал убедительного преимущества над простой ItemKNN по совокупности метрик.